In [1]:
# Setup
import sys
sys.path.append('em43_python')

from em43_notebook import EM43Trainer, quick_train_ga, quick_train_rs, compare_methods, compare_diversity_methods
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [6]:
# Step 1: Train separately
trainer = EM43Trainer(method='ga', generations=50, pop_size=5000, verbose=True)
genome, fitness = trainer.train(inputs=[1,2,3,4,5,6,7,8,9,10], targets=[2,4,6,8,10,12,14,16,18,20])
print(f"Trained! Fitness: {fitness:.3f}")

# Step 2: Infer separately
test_inputs = [1,2,3,4,5]
outputs = trainer.infer(test_inputs)
print(f"Inference results: {test_inputs} → {outputs}")

# Step 3: Evaluate separately
expected = [2,4,6,8,10]
metrics = trainer.evaluate(test_inputs, expected)
print("Evaluation metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.3f}")


Training GA for 50 generations...
Population: 5000, Program length: 10
Inputs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Targets: [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
Training complete! Best fitness: -0.601
Trained! Fitness: -0.601
Inference: [1, 2, 3, 4, 5] → [2, 4, 5, 8, 9]
Inference results: [1, 2, 3, 4, 5] → [2, 4, 5, 8, 9]
Inference: [1, 2, 3, 4, 5] → [2, 4, 5, 8, 9]
Evaluation metrics:
  avg_error: 0.400
  max_error: 1.000
  success_rate: 0.600
  accuracy: 0.600
  fitness: -0.601
Evaluation metrics:
  avg_error: 0.400
  max_error: 1.000
  success_rate: 0.600
  accuracy: 0.600
  fitness: -0.601


In [7]:
fitness_curve = trainer.get_fitness_curve()
print(fitness_curve)

[-3.1040000915527344, -3.1040000915527344, -2.7019999027252197, -1.4010000228881836, -1.4010000228881836, -1.4010000228881836, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.2020000219345093, -1.0010000467300415, -1.0010000467300415, -1.0010000467300415, -1.0010000467300415, -1.0010000467300415, -1.0010000467300415, -1.0010000467300415, -0.9010000228881836, -0.9010000228881836, -0.9010000228881836, -0.9010000228881836, -0.9010000228881836, -0.9010000228881836, -0.9010000228881836, -0.9010000228881836, -0.9010000228881836, -0.703000009059906, -0.703000009059906, -0.703000009059906, -0.7009999752044678, -0.700999975204

In [5]:
# Compare performance across multiple seeds
results = []
inputs = [1, 2, 3, 4, 5,6,7,8,9,10]
targets = [2, 4, 6, 8, 10,12,14,16,18,20]
test_inputs = [11,12,13,14,15]
test_targets = [22,24,26,28,30]

for seed in range(5):  # 5 runs for demonstration
    # GA run
    trainer_ga = EM43Trainer(method='ga', generations=50, pop_size=1000, seed=seed, verbose=False)
    _, fitness_ga = trainer_ga.train(inputs, targets)
    metrics_ga = trainer_ga.evaluate(test_inputs, test_targets)
    
    # Random Search run
    trainer_rs = EM43Trainer(method='random_search', generations=30, pop_size=500, seed=seed, verbose=False)
    _, fitness_rs = trainer_rs.train(inputs, targets)
    metrics_rs = trainer_rs.evaluate(test_inputs, test_targets)
    
    results.append({
        'seed': seed,
        'ga_fitness': fitness_ga,
        'rs_fitness': fitness_rs,
        'ga_accuracy': metrics_ga['accuracy'],
        'rs_accuracy': metrics_rs['accuracy']
    })
    
    print(f"Seed {seed}: GA={fitness_ga:.3f}, RS={fitness_rs:.3f}")

df = pd.DataFrame(results)
print("\nSummary Statistics:")
print(df[['ga_fitness', 'rs_fitness', 'ga_accuracy', 'rs_accuracy']].describe())


Seed 0: GA=-1.201, RS=-2.502
Seed 1: GA=-1.702, RS=-2.502
Seed 2: GA=-1.903, RS=-1.703
Seed 3: GA=-1.201, RS=-2.402
Seed 4: GA=-1.001, RS=-2.502

Summary Statistics:
       ga_fitness  rs_fitness  ga_accuracy  rs_accuracy
count    5.000000    5.000000          5.0     5.000000
mean    -1.401600   -2.322200          0.0     0.080000
std      0.381642    0.348841          0.0     0.178885
min     -1.903000   -2.502000          0.0     0.000000
25%     -1.702000   -2.502000          0.0     0.000000
50%     -1.201000   -2.502000          0.0     0.000000
75%     -1.201000   -2.402000          0.0     0.000000
max     -1.001000   -1.703000          0.0     0.400000


In [ ]:
# Demonstrate stochastic tournament selection for better diversity
inputs = [1, 2, 3, 4, 5]
targets = [2, 4, 6, 8, 10]

print("🔄 Comparing Standard vs Stochastic Tournament GA:")
print("=" * 60)

# Standard GA (greedy tournament selection)
print("\n1️⃣ Standard GA (greedy tournament):")
trainer_standard = EM43Trainer(
    method='ga',
    generations=50,
    pop_size=2000,
    enable_stochastic_tournament=False,  # Standard tournament
    verbose=True
)
genome_std, fitness_std = trainer_standard.train(inputs, targets)
metrics_std = trainer_standard.evaluate([6, 7, 8], [12, 14, 16])

print("\n2️⃣ Stochastic Tournament GA (less greedy):")
trainer_stochastic = EM43Trainer(
    method='ga', 
    generations=50,
    pop_size=2000,
    enable_stochastic_tournament=True,   # Stochastic tournament
    initial_temperature=2.5,             # Start exploratory
    cooling_rate=0.97,                   # Cool slowly
    min_temperature=0.1,                 # Maintain some diversity
    verbose=True
)
genome_stoch, fitness_stoch = trainer_stochastic.train(inputs, targets)
metrics_stoch = trainer_stochastic.evaluate([6, 7, 8], [12, 14, 16])

print("\n📊 Comparison Results:")
print(f"Standard GA     - Final Fitness: {fitness_std:.3f}, Test Accuracy: {metrics_std['accuracy']:.3f}")
print(f"Stochastic GA   - Final Fitness: {fitness_stoch:.3f}, Test Accuracy: {metrics_stoch['accuracy']:.3f}")

# Plot fitness curves
std_curve = trainer_standard.get_fitness_curve()
stoch_curve = trainer_stochastic.get_fitness_curve()

plt.figure(figsize=(10, 6))
plt.plot(std_curve, label='Standard Tournament', linewidth=2)
plt.plot(stoch_curve, label='Stochastic Tournament', linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Best Fitness')
plt.title('Learning Curves: Standard vs Stochastic Tournament')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 Note: Stochastic tournament may achieve similar or better performance")
print("   while maintaining more diversity (especially beneficial for complex tasks)")


In [ ]:
# Use the new compare_diversity_methods function
from em43_notebook import compare_diversity_methods, quick_train_ga

print("🎯 Quick Diversity Comparison Example:")
print("=" * 50)

# Quick single runs
print("\n1️⃣ Quick single runs:")
print("Standard GA:")
_, fitness_std = quick_train_ga([1,2,3], [3,6,9], generations=30, enable_stochastic_tournament=False)
print(f"Result: {fitness_std:.3f}")

print("\nStochastic GA:")
_, fitness_stoch = quick_train_ga([1,2,3], [3,6,9], generations=30, enable_stochastic_tournament=True)
print(f"Result: {fitness_stoch:.3f}")

print("\n2️⃣ Statistical comparison across multiple runs:")
# Statistical comparison
comparison = compare_diversity_methods(
    inputs=[1, 2, 3, 4],
    targets=[4, 8, 12, 16],
    generations=25,
    n_runs=8,  # Use 8 runs for statistical significance
    pop_size=1000,
    initial_temperature=2.0,
    cooling_rate=0.98
)

# Analyze results
import numpy as np
std_mean = np.mean(comparison['standard_ga'])
std_std = np.std(comparison['standard_ga'])
stoch_mean = np.mean(comparison['stochastic_ga'])
stoch_std = np.std(comparison['stochastic_ga'])

print(f"\n📈 Statistical Summary:")
print(f"Standard GA:   {std_mean:.3f} ± {std_std:.3f}")
print(f"Stochastic GA: {stoch_mean:.3f} ± {stoch_std:.3f}")

# Determine which performed better
if stoch_mean > std_mean:
    improvement = ((stoch_mean - std_mean) / abs(std_mean)) * 100
    print(f"✅ Stochastic GA improved by {improvement:.1f}%")
else:
    decline = ((std_mean - stoch_mean) / abs(std_mean)) * 100
    print(f"⚠️  Standard GA was {decline:.1f}% better (stochastic may need tuning)")

print("\n💡 Tips for tuning stochastic tournament:")
print("   • Higher initial_temperature = more exploration")
print("   • Lower cooling_rate = slower convergence, more diversity")
print("   • Higher min_temperature = maintains diversity longer")


In [ ]:
# Quick training functions (now with diversity options)
print("Quick GA training (standard):")
genome_ga, fitness_ga = quick_train_ga([1,2,3], [3,6,9], generations=20)
print(f"GA result: {fitness_ga:.3f}")

print("\nQuick GA training (stochastic tournament):")
genome_ga_stoch, fitness_ga_stoch = quick_train_ga([1,2,3], [3,6,9], generations=20, enable_stochastic_tournament=True)
print(f"GA (stochastic) result: {fitness_ga_stoch:.3f}")

print("\nQuick Random Search training:")
genome_rs, fitness_rs = quick_train_rs([1,2,3], [3,6,9], generations=20)
print(f"RS result: {fitness_rs:.3f}")

# Quick comparison
print("\nMethod comparison:")
comparison = compare_methods([1,2,3,4], [4,8,12,16], generations=25, n_runs=3)

print(f"GA average: {np.mean(comparison['ga']):.3f}")
print(f"RS average: {np.mean(comparison['rs']):.3f}")
